# Figure assets export (no training, CPU is enough, about 3 to 5 minutes)

This notebook saves **real images and real model outputs** for the paper figures:
the tumor probability map, the cropped region, the class probabilities of the proposed model,
the tumor hiding examples and tumor transplantation examples. It also copies the result tables.

**How to run**
1. Accelerator: **None** (GPU not needed).
2. Add Input: the Figshare PNG dataset, your **Run 6 results** dataset, and the datasets that hold the **model checkpoints** (`.pt` files).
3. Click **Run All**.
4. Download `figure_assets.zip` from the Output panel (right side, /kaggle/working) and send it to Claude.

In [ ]:
import os, json, math, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image

INPUT = Path(os.environ.get("ASSET_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("ASSET_WORK", "/kaggle/working")); OUT = WORK / "figure_assets"
shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir(parents=True)
IMG, K = 224, 3
CLASSES = ["glioma", "meningioma", "pituitary"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
SEG_NAME, CLS_NAME, MARGIN, MIN_PIX, N_PER_CLASS, N_TRANSPLANT = "MTL-Seg", "ROI-m06-Transplant", 0.6, 20, 8, 12

def find_one(name, required=True):
    hits = sorted(INPUT.rglob(name), key=lambda p: p.stat().st_size, reverse=True)
    if not hits and required:
        raise FileNotFoundError(f"'{name}' not found under {INPUT}. Add the dataset that contains it (see the list at the top).")
    return hits[0] if hits else None

SPLIT, MASKF = find_one("split_3class.csv"), find_one("figshare_masks_224.npz")
CK = {(n, f): find_one(f"{n}_s0_f{f}.pt") for n in (SEG_NAME, CLS_NAME) for f in range(1, 6)}
print("split file :", SPLIT); print("mask file  :", MASKF); print("checkpoints:", len(CK), "found")

_IDX = {}
for p in INPUT.rglob("*.png"):
    _IDX.setdefault(p.name, str(p))
def resolve(path):
    return path if os.path.exists(path) else _IDX.get(Path(path).name)

df = pd.read_csv(SPLIT); df["path"] = df.filepath.map(resolve); df = df.dropna(subset=["path"]).reset_index(drop=True)
z = np.load(MASKF, allow_pickle=True)
MASKS = {str(s): np.unpackbits(b)[:IMG * IMG].reshape(IMG, IMG).astype(bool) for s, b in zip(z["stems"], z["bits"])}
df = df[df.path.map(lambda p: Path(p).stem in MASKS)].reset_index(drop=True)
print("slices with image and mask:", len(df), "| test slices:", int((df.split == "test").sum()))

def load_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((IMG, IMG), Image.BICUBIC), np.uint8)
def native(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), np.uint8)
def mask_of(path): return MASKS[Path(path).stem]
def save(a, name): Image.fromarray(a).save(OUT / name)

# ---- identical to the Run 6 notebook -----------------------------------------------------------
def brain_mask(u8):
    _, th = cv2.threshold(cv2.GaussianBlur(u8, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(th)
    if n <= 1:
        return th > 0
    m = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(m); cv2.drawContours(filled, cnts, -1, 1, -1)
    return filled > 0

HIDE_KER = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
def shifted_region(region, bm, rng, tries=200):
    ys, xs = np.where(region)
    if len(ys) == 0:
        return None
    near = cv2.dilate(region.astype(np.uint8), HIDE_KER, iterations=2).astype(bool)
    for _ in range(tries):
        dy, dx = (int(v) for v in rng.integers(-IMG // 2, IMG // 2, 2))
        y2, x2 = ys + dy, xs + dx
        if y2.min() < 0 or x2.min() < 0 or y2.max() >= IMG or x2.max() >= IMG:
            continue
        r = np.zeros_like(region); r[y2, x2] = True
        if not (r & near).any() and bm[r].mean() >= 0.95:
            return dy, dx, r
    return None

def hide_versions(img, mk, rng):
    reg = cv2.dilate(mk.astype(np.uint8), HIDE_KER).astype(bool); bm = brain_mask(img)
    inp = cv2.inpaint(img, reg.astype(np.uint8) * 255, 7, cv2.INPAINT_TELEA)
    sh = shifted_region(reg, bm, rng)
    if sh is not None:
        dy, dx, rr = sh; ys, xs = np.where(reg)
        swap = img.copy(); swap[ys, xs] = img[ys + dy, xs + dx]
        rnd = cv2.inpaint(img, rr.astype(np.uint8) * 255, 7, cv2.INPAINT_TELEA)
    else:
        swap, rnd, rr = inp, img, np.zeros_like(reg)
    interior = cv2.erode(bm.astype(np.uint8), np.ones((17, 17), np.uint8)).astype(bool)
    skull = img.copy(); skull[interior | reg] = 0
    return {"region": reg, "inpaint": inp, "swap": swap, "random": rnd, "skull": skull, "spot": rr, "has_control": sh is not None}

class GuidedConvNeXt(nn.Module):
    def __init__(self, num_classes=K, seg=False):
        super().__init__()
        self.use_seg = seg
        self.backbone = timm.create_model("convnext_tiny", pretrained=False, features_only=True, out_indices=(1, 2, 3))
        c1, c2, c3 = self.backbone.feature_info.channels()
        self.cam_id = nn.Identity(); self.cam_id2 = nn.Identity()
        if seg:
            self.lat3, self.lat2, self.lat1 = nn.Conv2d(c3, 128, 1), nn.Conv2d(c2, 128, 1), nn.Conv2d(c1, 128, 1)
            self.seg_head = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(), nn.Conv2d(64, 1, 1))
        self.head_norm = nn.LayerNorm(c3)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(c3, num_classes))
    def forward(self, x, return_seg=False):
        f1, f2, f3 = self.backbone(x); seg = None
        if self.use_seg:
            d = self.lat3(f3)
            d = self.lat2(f2) + F.interpolate(d, size=f2.shape[-2:], mode="bilinear", align_corners=False)
            d = self.lat1(f1) + F.interpolate(d, size=f1.shape[-2:], mode="bilinear", align_corners=False)
            seg = self.seg_head(d)
        logits = self.head(self.head_norm(f3.mean((2, 3))))
        return (logits, seg) if return_seg else logits

def box_from_mask(mask, min_pixels=MIN_PIX):
    m = np.asarray(mask, np.uint8)
    if m.sum() < min_pixels:
        return None
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    if st[k, cv2.CC_STAT_AREA] < min_pixels:
        return None
    x, y, w, h = (int(v) for v in st[k, :4])
    return x, y, w, h, lab == k

def window(box, margin=MARGIN):
    x, y, w, h, _ = box; s = max(max(w, h) * (1 + 2 * margin), 32); cx, cy = x + w / 2, y + h / 2
    return int(cx - s / 2), int(cy - s / 2), int(cx + s / 2), int(cy + s / 2)
def crop_roi(img, box, margin=MARGIN):
    return np.asarray(Image.fromarray(img).crop(window(box, margin)).resize((IMG, IMG), Image.BICUBIC), np.uint8)

def transplant(rec_img, rec_hidden, rec_bm, don_img, don_mask, don_bm, rng):
    ys, xs = np.where(don_mask); bys, bxs = np.where(don_bm); rys, rxs = np.where(rec_bm)
    if len(ys) == 0 or len(rys) == 0:
        return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    pad = 4; y0, x0 = max(y0 - pad, 0), max(x0 - pad, 0); y1, x1 = min(y1 + pad, IMG), min(x1 + pad, IMG)
    patch = don_img[y0:y1, x0:x1].astype(np.float32)
    alpha = cv2.GaussianBlur(cv2.dilate(don_mask.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(np.float32), (7, 7), 2)[y0:y1, x0:x1]
    scale = float(np.clip((np.ptp(rys) + 1) / (np.ptp(bys) + 1), 0.7, 1.4))
    h, w = max(int(round((y1 - y0) * scale)), 3), max(int(round((x1 - x0) * scale)), 3)
    patch = cv2.resize(patch, (w, h), interpolation=cv2.INTER_LINEAR); alpha = cv2.resize(alpha, (w, h), interpolation=cv2.INTER_LINEAR)
    med_d = np.median(don_img[don_bm & ~don_mask]) + 1e-3; med_r = np.median(rec_img[rec_bm]) + 1e-3
    patch = patch * float(np.clip(med_r / med_d, 0.6, 1.6))
    ry = (ys.mean() - bys.min()) / max(np.ptp(bys), 1); rx = (xs.mean() - bxs.min()) / max(np.ptp(bxs), 1)
    if rng.random() < 0.5:
        rx = 1 - rx
    cy = rys.min() + ry * np.ptp(rys) + rng.normal(0, 3); cx = rxs.min() + rx * np.ptp(rxs) + rng.normal(0, 3)
    ty, tx = int(round(cy - h / 2)), int(round(cx - w / 2))
    if ty < 0 or tx < 0 or ty + h > IMG or tx + w > IMG:
        return None
    if rec_bm[ty:ty + h, tx:tx + w][alpha > 0.5].mean() < 0.9:
        return None
    out = rec_hidden.astype(np.float32).copy(); region = out[ty:ty + h, tx:tx + w]
    out[ty:ty + h, tx:tx + w] = alpha * patch + (1 - alpha) * region
    mk = np.zeros((IMG, IMG), bool); mk[ty:ty + h, tx:tx + w] = alpha > 0.5
    return np.clip(out, 0, 255).astype(np.uint8), mk, (tx, ty, w, h), (int(x0), int(y0), int(x1 - x0), int(y1 - y0))
# -------------------------------------------------------------------------------------------------
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def to_batch(arrs):
    x = torch.from_numpy(np.stack(arrs)).to(DEV).float().div_(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - torch.tensor(MEAN, device=DEV).view(1, 3, 1, 1)) / torch.tensor(STD, device=DEV).view(1, 3, 1, 1)
def load(name, fold, seg):
    m = GuidedConvNeXt(seg=seg).to(DEV)
    sd = torch.load(CK[(name, fold)], map_location=DEV); m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=True)
    return m.eval()

In [ ]:
# ---- 1. Inference examples: real tumor map, real crop, real probabilities (test set, never used for training) ----
rng = np.random.default_rng(7); cand = []
test = df[df.split == "test"]
for c in CLASSES:
    sub = test[test.label4 == c]
    ok = [p for p in sub.path if 0.006 <= mask_of(p).mean() <= 0.07]
    ok = ok if len(ok) >= N_PER_CLASS else list(sub.path)
    cand += [(c, p) for p in rng.permutation(ok)[:N_PER_CLASS]]
arrs = [load_u8(p) for _, p in cand]
maps, probs, found = [], [], []
with torch.no_grad():
    for f in range(1, 6):
        seg = load(SEG_NAME, f, True); cls = load(CLS_NAME, f, False)
        _, sl = seg(to_batch(arrs), return_seg=True)
        mp = torch.sigmoid(F.interpolate(sl.float(), (IMG, IMG), mode="bilinear", align_corners=False))[:, 0].cpu().numpy()
        pr = np.full((len(arrs), K), 1 / K, np.float32); fd = np.zeros(len(arrs), bool)
        for i, (a, m_) in enumerate(zip(arrs, mp)):
            b = box_from_mask(m_ >= 0.5)
            if b is not None:
                fd[i] = True; pr[i] = torch.softmax(cls(to_batch([crop_roi(a, b)])).float(), 1)[0].cpu().numpy()
        maps.append(mp); probs.append(pr); found.append(fd); del seg, cls
        print(f"fold {f} done")
P = np.mean(probs, 0); meta = []
for i, ((c, p), a) in enumerate(zip(cand, arrs)):
    tag = f"inf_{i:02d}"; nat = native(p); mk = mask_of(p); mp = maps[0][i]; b = box_from_mask(mp >= 0.5)
    save(a, f"{tag}_slice224.png"); save(nat, f"{tag}_native.png"); save((mk * 255).astype(np.uint8), f"{tag}_expertmask.png")
    save((np.clip(mp, 0, 1) * 255).astype(np.uint8), f"{tag}_probmap_fold1.png")
    save((np.clip(np.mean([m[i] for m in maps], 0), 0, 1) * 255).astype(np.uint8), f"{tag}_probmap_mean.png")
    row = {"tag": tag, "class": c, "stem": Path(p).stem, "native_size": list(nat.shape), "probs_5fold_mean": [float(v) for v in P[i]],
           "probs_fold1": [float(v) for v in probs[0][i]], "found_per_fold": [bool(fd[i]) for fd in found], "pred": CLASSES[int(P[i].argmax())],
           "mask_area": float(mk.mean())}
    if b is not None:
        x0, y0, x1, y1 = window(b); row["window_fold1"] = [x0, y0, x1, y1]; row["box_fold1"] = [int(v) for v in b[:4]]
        save(crop_roi(a, b), f"{tag}_crop224_fold1.png"); save((b[4] * 255).astype(np.uint8), f"{tag}_component_fold1.png")
        sx, sy = nat.shape[1] / IMG, nat.shape[0] / IMG
        save(np.asarray(Image.fromarray(nat).crop((int(x0 * sx), int(y0 * sy), int(x1 * sx), int(y1 * sy))).resize((448, 448), Image.BICUBIC)), f"{tag}_crop_native.png")
    hv = hide_versions(a, mk, np.random.default_rng(100 + i)); row["has_control"] = bool(hv["has_control"])
    for k in ("swap", "random", "skull", "inpaint"):
        save(hv[k], f"{tag}_hide_{k}.png")
    save((hv["region"] * 255).astype(np.uint8), f"{tag}_hide_region.png"); save((hv["spot"] * 255).astype(np.uint8), f"{tag}_hide_spot.png")
    meta.append(row)
json.dump(meta, open(OUT / "inference_examples.json", "w"), indent=1)
print(pd.DataFrame(meta)[["tag", "class", "pred", "has_control"]].assign(conf=[max(r["probs_5fold_mean"]) for r in meta]).to_string(index=False))

In [ ]:
# ---- 2. Tumor transplantation examples (training patients of fold 1) ----
rng = np.random.default_rng(11); train = df[(df.split != "test") & (df.split != "fold1")]; tmeta = []; tries = 0
while len(tmeta) < N_TRANSPLANT and tries < 600:
    tries += 1
    cr, cd = rng.choice(CLASSES, 2, replace=False)
    pr_, pd_ = rng.choice(train[train.label4 == cr].path.values), rng.choice(train[train.label4 == cd].path.values)
    ri, di = load_u8(pr_), load_u8(pd_); rm, dm = mask_of(pr_), mask_of(pd_)
    if not (0.006 <= dm.mean() <= 0.07):
        continue
    hid = hide_versions(ri, rm, rng)["inpaint"]
    res = transplant(ri, hid, brain_mask(ri), di, dm, brain_mask(di), rng)
    if res is None:
        continue
    comp, cm, place, src = res; b = box_from_mask(cm); tag = f"tt_{len(tmeta):02d}"
    if b is None:
        continue
    for a, n in [(ri, "recipient"), (hid, "recipient_inpainted"), (di, "donor"), (comp, "composite"), (crop_roi(comp, b), "composite_crop224")]:
        save(a, f"{tag}_{n}.png")
    for a, n in [(rm, "recipient_mask"), (dm, "donor_mask"), (cm, "composite_mask")]:
        save((a * 255).astype(np.uint8), f"{tag}_{n}.png")
    save(native(pr_), f"{tag}_recipient_native.png"); save(native(pd_), f"{tag}_donor_native.png")
    tmeta.append({"tag": tag, "recipient_class": str(cr), "donor_class": str(cd), "recipient": Path(pr_).stem, "donor": Path(pd_).stem,
                  "placed_xywh": [int(v) for v in place], "donor_patch_xywh": list(src), "window": list(window(b))})
json.dump(tmeta, open(OUT / "transplant_examples.json", "w"), indent=1)
print("transplant examples saved:", len(tmeta))

In [ ]:
# ---- 3. Copy result tables and the final heat map figures, then zip ----
tdir = next((p.parent for p in INPUT.rglob("table_reliance.csv")), None)
if tdir:
    (OUT / "tables").mkdir(exist_ok=True)
    for f in tdir.glob("*.csv"):
        shutil.copy(f, OUT / "tables" / f.name)
    print("tables copied:", len(list((OUT / "tables").glob("*.csv"))))
else:
    print("result tables not found (add the Run 6 results dataset)")
(OUT / "figures").mkdir(exist_ok=True); budget = 8e6
for pat in ["fig_xai_hires_main*.png", "fig_xai_hires_external*.png", "fig_xai_hires_other*.png"]:
    for f in sorted(INPUT.rglob(pat))[:1]:
        if f.stat().st_size < budget:
            shutil.copy(f, OUT / "figures" / f.name); budget -= f.stat().st_size
zp = WORK / "figure_assets.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")):
        if f.is_file(): zf.write(f, f.relative_to(WORK))
print(f"\nDONE. figure_assets.zip = {zp.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output panel (right side, /kaggle/working) and send it to Claude.")